# Assignment: Recommendation System
###Anime Recommendation System using Cosine Similarity

# 1. Introduction
Recommendation systems are one of the most widely used applications of machine learning and data
science. They help users discover relevant products, movies, music, anime, and other content based on
similarity measures and user preferences.
In this assignment, an anime recommendation system is developed using Cosine Similarity. The
recommendation system suggests anime titles similar to a target anime based on features such as:
Genre

Type

Number of Episodes

Average Rating

Community Members

The project uses a content-based filtering approach, where similarity between anime titles is calculated
using feature vectors.

# 2. Objective
The objective of this assignment is:
1. To preprocess the anime dataset.
2. To perform feature engineering.
3. To compute cosine similarity between anime titles.
4. To build a recommendation system that recommends similar anime.
5. To analyze system performance and discuss improvements.

# 3. Dataset Description
The dataset contains information about different anime titles.
Feature Description:-
anime_id Unique ID of each anime

name Anime title

genre Anime genres

type Broadcast type such as TV, Movie, OVA

episodes Number of episodes

rating Average user rating

members Number of community members

# Dataset Size:
###Total Rows: 12,294
###Total Columns: 7

# 4. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')


# 5. Load the Dataset


In [2]:
# Load dataset
anime = pd.read_csv('/content/anime.csv')
# Display first 5 rows
print(anime.head())


   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  


# 6. Exploratory Data Analysis (EDA)

In [3]:
# Dataset Infobox
print(anime.info())
# Check Missing Values
print(anime.isnull().sum())
# Statistical Summary
print(anime.describe())
# Shape of Data
print('Dataset Shape:', anime.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
None
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64
           anime_id        rating       members
count  12294.000000  12064.000000  1.229400e+04
mean   14058.221653      6.473902  1.807134e+04
std    11455.294701      1.026746  5.482068e+04
min        1.000000      1.670000  5.000000e+00
25%     3484.250000      5.880000  2.250000e+02
50%    10260.500000      6.570000  1.550000e+03
75%    2479

# 7. Data Preprocessing
### 7.1 Handle Missing Values

In [4]:

# Fill missing genre values
anime['genre'] = anime['genre'].fillna('Unknown')
# Fill missing type values
anime['type'] = anime['type'].fillna('Unknown')
# Fill missing ratings with median
anime['rating'] = anime['rating'].fillna(anime['rating'].median())
# Remove rows with missing names
anime = anime.dropna(subset=['name'])

### 7.2 Convert Episodes Column
Some episode values contain non-numeric entries such as "Unknown".

In [5]:
anime['episodes'] = pd.to_numeric(anime['episodes'], errors='coerce')
# Fill missing episodes with median
anime['episodes'] = anime['episodes'].fillna(anime['episodes'].median())

# 8. Feature Extraction
To compute cosine similarity, categorical and numerical features must be converted into numerical form.
The following features are used:

Genre

Type

Rating

Episodes

Members


# 8.1 Genre Vectorization
###The genre column is converted into a text vector using CountVectorizer.

In [6]:
vectorizer = CountVectorizer(tokenizer=lambda x: x.split(','))
genre_matrix = vectorizer.fit_transform(anime['genre'])

# 8.2 Numerical Feature Scaling
###Numerical features are normalized using MinMaxScaler.

In [7]:
scaler = MinMaxScaler()
numerical_features = scaler.fit_transform(
anime[['rating', 'episodes', 'members']]
)

#8.3 Combine Features
###Genre vectors and numerical features are combined.

In [8]:
from scipy.sparse import hstack
feature_matrix = hstack([genre_matrix, numerical_features])

# 9. Compute Cosine Similarity
###Cosine similarity measures similarity between feature vectors.
The cosine similarity formula is:
Cosine Similarity =
Cosine Similarity =

A ⋅ B / ∣∣A∣∣∣∣B∣∣

In [9]:
cosine_sim = cosine_similarity(feature_matrix)
print(cosine_sim.shape)

(12294, 12294)


# 10. Build Recommendation Function


In [11]:
# Create index mapping
indices = pd.Series(anime.index, index=anime['name']).drop_duplicates()
def recommend_anime(title, threshold=0.5, top_n=10):
    # Check whether anime exists
    if title not in indices:
        return 'Anime not found in dataset.'
    # Get anime index
    idx = indices[title]
    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Sort similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Remove self similarity
    sim_scores = sim_scores[1:]
    # Apply threshold
    sim_scores = [x for x in sim_scores if x[1] >= threshold]
    # Select top recommendations
    sim_scores = sim_scores[:top_n]
    # Get anime indices
    anime_indices = [i[0] for i in sim_scores]
    # Create recommendation dataframe
    recommendations = anime[['name', 'genre', 'rating']].iloc[anime_indices]
    recommendations['Similarity Score'] = [i[1] for i in sim_scores]
    return recommendations

# 11. Test the Recommendation System

In [12]:
recommend_anime('Naruto', threshold=0.4, top_n=5)

,name,genre,rating,Similarity Score
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94,0.997027
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53,0.969302
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50,0.969183
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",8.03,0.968273
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58,0.962834


# 12. Experiment with Threshold Values
###Threshold values affect the recommendation quality.
Threshold = 0.3

In [13]:
recommend_anime('Death Note', threshold=0.3)


,name,genre,rating,Similarity Score
778,Death Note Rewrite,"Mystery, Police, Psychological, Supernatural, ...",7.84,0.935399
144,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.41,0.871989
833,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.81,0.754358
2691,Yakushiji Ryouko no Kaiki Jikenbo,"Mystery, Police, Supernatural",7.19,0.743099
6323,Saint Luminous Jogakuin,"Mystery, Psychological, Supernatural",6.17,0.735591
10785,"Yakushiji Ryouko no Kaiki Jikenbo: Hamachou, V...","Mystery, Police, Supernatural",5.97,0.733768
49,Boku dake ga Inai Machi,"Mystery, Psychological, Seinen, Supernatural",8.65,0.718644
981,Mousou Dairinin,"Drama, Mystery, Police, Psychological, Superna...",7.74,0.716540
541,Shiki,"Mystery, Supernatural, Thriller, Vampire",7.99,0.696967
1325,Shinreigari: Ghost Hound,"Mystery, Psychological, Sci-Fi, Supernatural",7.59,0.668318


# Observation
* Larger recommendation list
* Less strict similarity
* More diverse recommendations

# Threshold = 0.7


In [14]:
recommend_anime('Death Note', threshold=0.7)

,name,genre,rating,Similarity Score
778,Death Note Rewrite,"Mystery, Police, Psychological, Supernatural, ...",7.84,0.935399
144,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.41,0.871989
833,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.81,0.754358
2691,Yakushiji Ryouko no Kaiki Jikenbo,"Mystery, Police, Supernatural",7.19,0.743099
6323,Saint Luminous Jogakuin,"Mystery, Psychological, Supernatural",6.17,0.735591
10785,"Yakushiji Ryouko no Kaiki Jikenbo: Hamachou, V...","Mystery, Police, Supernatural",5.97,0.733768
49,Boku dake ga Inai Machi,"Mystery, Psychological, Seinen, Supernatural",8.65,0.718644
981,Mousou Dairinin,"Drama, Mystery, Police, Psychological, Superna...",7.74,0.716540


# Observation
* Smaller recommendation list
* Highly similar anime
* Better recommendation precision

# 13. Performance Analysis
###Advantages
1. Simple and easy to implement
2. Fast recommendation generation
3. Works well for content similarity
4. Does not require user interaction history
###Limitations
1. Cannot capture changing user preferences
2. Recommendations depend heavily on feature quality
3. Cold-start problem for new anime entries
4. Limited personalization

# 14. Areas of Improvement
The recommendation system can be improved using:
* Hybrid Recommendation Systems
* Deep Learning Models
* User Rating History
* Collaborative Filtering
* Matrix Factorization
* Word Embeddings for Genre Analysis
* TF-IDF Vectorization
* Real-time Recommendation Updates

# 16. Conclusion
###In this assignment, a content-based anime recommendation system was successfully implemented using
#cosine similarity.
The project included:
1. Data preprocessing
2. Feature extraction
3. Numerical normalization
4. Cosine similarity computation
5. Recommendation generation
6. Threshold experimentation
7. The system recommends anime titles with similar genres and characteristics effectively.
8. This project demonstrates the practical application of recommendation systems in real-world entertainment platforms.

# 17. Interview Questions
###Q1. Explain the difference between user-based and item-based collaborative filtering.

# User-Based Collaborative Filtering
User-based collaborative filtering recommends items based on similar users.
Working:
1. Find users with similar preferences.
2. Recommend items liked by similar users.
###Example:
If User A and User B both like Naruto and Death Note, and User A also likes One Piece, then One Piece may
be recommended to User B.

###Advantages:
1. Personalized recommendations
2. Easy to understand

###Disadvantages:
1. Computationally expensive for large datasets
2. Sparsity problem
3. Item-Based Collaborative Filtering
4. Item-based collaborative filtering recommends items similar to the current item.

###Working:
Find similar items based on user ratings.
Recommend items similar to what the user already likes.

###Example:
If users who watched Naruto also watched Bleach, then Bleach may be recommended.

###Advantages:
1. Faster than user-based filtering
2. More scalable
3. Better for large systems

###Disadvantages:

1. Requires sufficient item interaction data
2. Difference Summary
3. User-Based Item-Based
4. Finds similar users Finds similar items
5. User-user similarity Item-item similarity
6. Less scalable More scalable
7. Dynamic recommendations Stable recommendations
8. Computationally expensive Computationally efficient


# Q2. What is collaborative filtering and how does it work?
# Collaborative filtering is a recommendation technique that recommends items based on the behavior and preferences of multiple users.
###It assumes that:
Users with similar interests are likely to prefer similar items.

# Types of Collaborative Filtering
1. User-Based Collaborative Filtering
Uses similarities between users.
Recommends items liked by similar users.
2. Item-Based Collaborative Filtering
Uses similarities between items.
Recommends items similar to previously liked items

###Steps in Collaborative Filtering
* Collect user-item interaction data.
* Compute similarity between users or items.
* Identify nearest neighbors.
* Generate recommendations.

###Advantages

* Personalized recommendations
* No need for detailed item features
* Effective in many applications

###Disadvantages

* Cold start problem
* Data sparsity
* Scalability challenges

